# Rec Systems II
# Content-Based Methods

This workbook is part of a series, listed below:
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20I%20-%20Baseline%20Methods.ipynb">Rec Systems I - Baseline Methods</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20II%20-%20Content%20Based.ipynb">Rec Systems II - Content Based</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20III%20-%20User-Based%20Collaborative%20Filtering.ipynb">Rec Systems III - User-Based Collaborative Filtering</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20IV%20-%20Item-Based%20Collaborative%20Filtering.ipynb">Rec Systems IV - Item-Based Collaborative Filtering</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20V%20-%20Matrix%20Factorization.ipynb">Rec Systems V - Matrix Factorization</a>

In this workbook, we'll stick with movies data, but use a Netflix dataset instead of MovieLens, as it has richer metadata such as descriptions and actors.

# Import Libraries and Data

In [1]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.metrics.pairwise import cosine_similarity

import warnings 
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('netflix_data/data.csv', encoding='latin-1')
df.head(2)

,Title,Genre,Tags,Languages,Director,Writer,Actors,View Rating,IMDb Score,Awards Received,Awards Nominated For,Boxoffice,Netflix Link,Summary,IMDb Votes,Image
0,Lets Fight Ghost,"Crime, Drama, Fantasy, Horror, Romance","Comedy Programmes,Romantic TV Comedies,Horror ...","Swedish, Spanish",Tomas Alfredson,John Ajvide Lindqvist,"Lina Leandersson, Kåre Hedebrant, Per Ragnar, ...",R,7.9,74.0,57.0,"$21,22,065",https://www.netflix.com/watch/81415947,A med student with a supernatural gift tries t...,205926.0,https://occ-0-4708-64.1.nflxso.net/dnm/api/v6/...
1,HOW TO BUILD A GIRL,Comedy,"Dramas,Comedies,Films Based on Books,British",English,Coky Giedroyc,Caitlin Moran,"Cleo, Paddy Considine, Beanie Feldstein, Dónal...",R,5.8,1.0,NaN,"$70,632",https://www.netflix.com/watch/81041267,"When nerdy Johanna moves to London, things get...",2838.0,https://occ-0-1081-999.1.nflxso.net/dnm/api/v6...


In [3]:
df = df.drop(columns=['View Rating', 'IMDb Score', 'Awards Received', \
                      'Awards Nominated For', 'Boxoffice', 'Netflix Link', \
                      'IMDb Votes', 'Image'])

# Preprocessing

In [4]:
def clean_text(text):
    if type(text)==str:
        text = text.lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

In [5]:
text_columns = df.columns[df.dtypes == 'object']
for col in text_columns:
    df[col] = df[col].apply(clean_text)

In [6]:
df.isnull().sum()

Title           0
Genre          25
Tags           37
Languages     159
Director     2304
Writer       1807
Actors        111
Summary         5
dtype: int64

In [7]:
df = df.fillna('unk')
df.isnull().sum()

Title        0
Genre        0
Tags         0
Languages    0
Director     0
Writer       0
Actors       0
Summary      0
dtype: int64

# Text Vectorization

A vector is a quantity that has both a magnitude and a direction. In NLP, we map text to vectors, such that the vectors for each class fall into distinct 'clouds' in Euclidean space.

### Bag of Words (BOW) Approach

Although the sequence of words provides meaning, many NLP approaches do not consider word order. One such method is the count-vectorizer, in which each word in a vocabulary is a feature, each sentence or document is a data instance, and the values for each cell represent the count of that word in the data instance. The feature (word) counts provide a row-wise vector reprsentation of each instance.

Vector similarity metrics then allow us to ask questions like 'which documents/sentences are most similar to the one in question?' We could also ask what documents are least similar, or even draw analogies from the location of the points in Euclidean space.

# Count-Vectorizer

In [8]:
text_data1 = df['Title'].tolist()
vectorizer1 = CountVectorizer()
cv_matrix1 = vectorizer1.fit_transform(text_data1)
dense_tfidf_matrix1 = cv_matrix1.todense()

text_data2 = df['Genre'].tolist()
vectorizer2 = CountVectorizer()
cv_matrix2 = vectorizer2.fit_transform(text_data2)
dense_tfidf_matrix2 = cv_matrix2.todense()

text_data3 = df['Tags'].tolist()
vectorizer3 = CountVectorizer()
cv_matrix3 = vectorizer3.fit_transform(text_data3)
dense_tfidf_matrix3 = cv_matrix3.todense()

text_data4 = df['Actors'].tolist()
vectorizer4 = CountVectorizer()
cv_matrix4 = vectorizer4.fit_transform(text_data4)
dense_tfidf_matrix4 = cv_matrix4.todense()

text_data5 = df['Summary'].tolist()
vectorizer5 = CountVectorizer()
cv_matrix5 = vectorizer5.fit_transform(text_data5)
dense_tfidf_matrix5 = cv_matrix5.todense()

In [9]:
compiled_matrix = hstack((cv_matrix1, cv_matrix2, cv_matrix3, cv_matrix4, cv_matrix5))
compiled_matrix = compiled_matrix.todense()
compiled_matrix = np.asarray(compiled_matrix)

In [10]:
print(df.shape)
print(compiled_matrix.shape)

(9403, 8)
(9403, 55739)


### Perform a Query

In [12]:
query_item_idx = np.random.choice(range(0,len(compiled_matrix),1))
df['Title'][query_item_idx]

'mesrine part 1 killer instinct'

In [13]:
result_df = df.copy()
result_df['cos_sim'] = np.nan

for i in tqdm(range(0,len(compiled_matrix))):
    cos_sim = cosine_similarity((compiled_matrix[query_item_idx], compiled_matrix[i]))
    result_df['cos_sim'][i] = cos_sim[1,0] 

100%|██████████| 9403/9403 [00:19<00:00, 492.35it/s]


In [14]:
result_df[result_df.index != query_item_idx].sort_values(by='cos_sim', ascending=False)[0:10]

,Title,Genre,Tags,Languages,Director,Writer,Actors,Summary,cos_sim
6311,mesrine part 2 public enemy 1,action biography crime drama thriller,french moviesthrillersinternational moviescrim...,french english,jeanfranois richet,abdel raouf dafri jeanfranois richet,samuel le bihan mathieu amalric vincent cassel...,paris police redouble their efforts against ga...,0.442634
5074,steve jobs,biography drama,dramas based on real lifebiographical dramasdr...,english,danny boyle,walter isaacson aaron sorkin,seth rogen jeff daniels kate winslet michael f...,by keying in on three crucial points in his ca...,0.405922
7099,zodiac,crime drama mystery thriller,crime filmsdramas20th century period piecesthr...,english,david fincher,james vanderbilt robert graysmith,anthony edwards jake gyllenhaal mark ruffalo r...,based on real events this chilling drama recou...,0.401530
8359,invictus,biography drama history sport,dramasdramas based on a bookdramas based on re...,english afrikaans maori zulu xhosa southern sotho,clint eastwood,john carlin anthony peckham,matt damon morgan freeman tony kgoroge patrick...,after the end of apartheid newly elected presi...,0.401148
7268,kill the irishman,biography crime drama,crime filmsdramas20th century period piecesgan...,english,jonathan hensleigh,rick porrello jonathan hensleigh jeremy walters,val kilmer ray stevenson christopher walken vi...,this true crime tale charts the rise and fall ...,0.400917
6693,the fifth estate,biography crime drama thriller,dramasdramas based on a bookbiographical drama...,english icelandic swahili arabic,bill condon,daniel domscheitberg luke harding josh singer ...,david thewlis peter capaldi anatole taubman al...,this factbased drama recounts the early days o...,0.397240
6035,the assassination of jesse james by the coward...,biography crime drama history western,dramasdramas based on contemporary literatured...,english danish,andrew dominik,andrew dominik ron hansen,dustin bollinger brad pitt brooklynn proulx ma...,after robert ford joins the most notorious gan...,0.393919
5836,serial killer 1,crime drama,dramascrime dramascrime filmsdramas based on a...,french,frdric tellier,frdric tellier david oelhoffen patricia touran...,raphal personnaz nathalie baye michel vuillerm...,the death of a young woman leads a french dete...,0.392754
5303,bugsy,biography crime drama,biographical dramascrime filmsdramas based on ...,english,barry levinson,dean jennings james toback,annette bening harvey keitel ben kingsley warr...,gangster bugsy siegel builds a gambling mecca ...,0.388457
7617,eat pray love,biography drama romance,romantic dramasdramasdramas based on a booktea...,english italian portuguese,ryan murphy,ryan murphy jennifer salt elizabeth gilbert,hadi subiyanto i gusti ayu puspawati julia rob...,after deciding to reshape her life after divor...,0.388140


# TF-IDF

Similar to how stopwords impede the ability to draw meaning from text, words that are common amongst sentences or documents are often less useful than infrequent words in terms of inferring the context. We would like to scale down the importance of common words based on how many 'documents' they appear in. One method is TF-IDF, in which the frequency of a word in a document (the term frequency) gets multiplied by the inverse of its frequency amongst documents.

$$TF-IDF \approx \frac{tf}{idf}$$

In practice, the IDF is usually calculated with the following adjustment:

$$idf = log \frac{N}{N(t)}$$

where $N$ is the number of documents and $N(t)$ is the number of documents in which the term in question appears. A variation is the smooth IDF, which avoids division by zero.

$$smooth ~idf = log \frac{N}{N(t)+1} + 1$$

The code is identical to above, except that we call upon the TfidfVectorizer instead of CountVectorizer.

In [15]:
text_data1 = df['Title'].tolist()
vectorizer1 = TfidfVectorizer()
tfidf_matrix1 = vectorizer1.fit_transform(text_data1)
dense_tfidf_matrix1 = tfidf_matrix1.todense()

text_data2 = df['Genre'].tolist()
vectorizer2 = TfidfVectorizer()
tfidf_matrix2 = vectorizer2.fit_transform(text_data2)
dense_tfidf_matrix2 = tfidf_matrix2.todense()

text_data3 = df['Tags'].tolist()
vectorizer3 = TfidfVectorizer()
tfidf_matrix3 = vectorizer3.fit_transform(text_data3)
dense_tfidf_matrix3 = tfidf_matrix3.todense()

text_data4 = df['Actors'].tolist()
vectorizer4 = TfidfVectorizer()
tfidf_matrix4 = vectorizer4.fit_transform(text_data4)
dense_tfidf_matrix4 = tfidf_matrix4.todense()

text_data5 = df['Summary'].tolist()
vectorizer5 = TfidfVectorizer()
tfidf_matrix5 = vectorizer5.fit_transform(text_data5)
dense_tfidf_matrix5 = tfidf_matrix5.todense()

In [16]:
compiled_matrix = hstack((tfidf_matrix1, tfidf_matrix2, tfidf_matrix3, tfidf_matrix4, tfidf_matrix5))
compiled_matrix = compiled_matrix.todense()
compiled_matrix = np.asarray(compiled_matrix)

In [17]:
print(df.shape)
print(compiled_matrix.shape)

(9403, 8)
(9403, 55739)


### Perform a Query

In [18]:
# using the same item as above
df['Title'][query_item_idx]

'mesrine part 1 killer instinct'

In [19]:
result_df = df.copy()
result_df['cos_sim'] = np.nan

for i in tqdm(range(0,len(compiled_matrix))):
    cos_sim = cosine_similarity((compiled_matrix[query_item_idx], compiled_matrix[i]))
    result_df['cos_sim'][i] = cos_sim[1,0] 

100%|██████████| 9403/9403 [00:18<00:00, 503.73it/s]


In [20]:
result_df[result_df.index != query_item_idx].sort_values(by='cos_sim', ascending=False)[0:10]

,Title,Genre,Tags,Languages,Director,Writer,Actors,Summary,cos_sim
6311,mesrine part 2 public enemy 1,action biography crime drama thriller,french moviesthrillersinternational moviescrim...,french english,jeanfranois richet,abdel raouf dafri jeanfranois richet,samuel le bihan mathieu amalric vincent cassel...,paris police redouble their efforts against ga...,0.466808
5565,midnight express,biography crime drama thriller,crime dramasbiographical dramasclassic dramasd...,english maltese french turkish,alan parker,oliver stone william hoffer billy hayes,bo hopkins brad davis irene miracle paolo bona...,arrested in 1970 after trying to smuggle two k...,0.267905
6693,the fifth estate,biography crime drama thriller,dramasdramas based on a bookbiographical drama...,english icelandic swahili arabic,bill condon,daniel domscheitberg luke harding josh singer ...,david thewlis peter capaldi anatole taubman al...,this factbased drama recounts the early days o...,0.258186
6902,captain phillips,adventure biography crime drama thriller,dramasthrillersdramas based on a bookdramas ba...,english somali,paul greengrass,stephan talty billy ray richard phillips,barkhad abdirahman barkhad abdi tom hanks cath...,in this exciting adventure based on true event...,0.253239
5690,the infiltrator,biography crime drama thriller,dramas based on real lifedramas based on a boo...,english spanish,brad furman,robert mazur ellen furman,leanne best bryan cranston daniel mays tom vau...,a canny federal agent navigates a deadly world...,0.251927
6753,freedom writers,biography crime drama,dramasdramas based on a booksocial issue drama...,english spanish,richard lagravenese,richard lagravenese erin gruwell freedom writers,imelda staunton hilary swank patrick dempsey s...,while her atrisk students are reading classics...,0.248621
7268,kill the irishman,biography crime drama,crime filmsdramas20th century period piecesgan...,english,jonathan hensleigh,rick porrello jonathan hensleigh jeremy walters,val kilmer ray stevenson christopher walken vi...,this true crime tale charts the rise and fall ...,0.248515
7094,the killer,action crime drama thriller,dramasgangster filmsasian action filmsinternat...,cantonese mandarin japanese,john woo,john woo,yunfat chow kong chu danny lee sally yeh,an assassin agrees to take on one final assign...,0.245787
5836,serial killer 1,crime drama,dramascrime dramascrime filmsdramas based on a...,french,frdric tellier,frdric tellier david oelhoffen patricia touran...,raphal personnaz nathalie baye michel vuillerm...,the death of a young woman leads a french dete...,0.245745
7972,devils knot,biography crime drama thriller,dramascourtroom dramasdramas based on a bookdr...,english,atom egoyan,paul harris boardman scott derrickson mara lev...,james hamrick alessandro nivola reese withersp...,frothing for vengeance after three 8yearolds a...,0.242661


The next workbook in the series looks at user-based collaborative filtering, and can be found <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20III%20-%20User-Based%20Collaborative%20Filtering.ipynb">here</a>.